# CampaignOptima: Phase 3 — Decision Science & Operations Research

**Objective:** Bridge predictive intelligence with marketing business operations:
1. **Cost-Benefit Threshold Optimization:** Maximize expected net profit by determining the empirical optimal cutoff $\tau^*$.
2. **Knapsack Budget Allocation (`scipy.optimize.linprog`):** Formulate linear programming to allocate limited marketing budgets across prospective customers while avoiding telemarketing burnout.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT_DIR = Path("..").resolve()
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

from src.data_loader import BankMarketingDataLoader
from src.features import CampaignOptimaFeaturePipeline
from src.models import ModelBenchmarkEngine
from src.optimization import ThresholdOptimizer, CampaignBudgetOptimizer

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

## 1. Load Data & Predict Continuous Probabilities

In [ ]:
loader = BankMarketingDataLoader()
df_raw = loader.load_raw_data()
X, y = loader.clean_and_prepare(df_raw)
X_train, X_test, y_train, y_test = loader.get_stratified_train_test_split(X, y)

pipeline = CampaignOptimaFeaturePipeline()
X_train_proc = pipeline.fit_transform(X_train)
X_test_proc = pipeline.transform(X_test)

from xgboost import XGBClassifier
model = XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.08, random_state=42)
model.fit(X_train_proc, y_train)
test_probs = model.predict_proba(X_test_proc)[:, 1]

## 2. Business Cost-Benefit Threshold Optimization
Unit Economics:
- Telemarketing cost per call: **$15.00**
- Net deposit conversion value: **$300.00**
- Theoretical break-even probability: $\tau = \frac{15}{300} = 0.05$ (5%)

In [ ]:
thresh_opt = ThresholdOptimizer(cost_per_call=15.0, conversion_revenue=300.0)
opt_results = thresh_opt.find_optimal_threshold(y_test.values, test_probs)

print(f"Optimal Operating Threshold (tau*): {opt_results['optimal_threshold']:.3f}")
print(f"Profit Uplift over Default (0.50): +${opt_results['profit_uplift']:,.2f} (+{opt_results['profit_uplift_pct']:.1f}%)")

### Plotting the Net Profit Curve vs Decision Threshold

In [ ]:
curve_df = opt_results["profit_curve_df"]

plt.figure(figsize=(9, 5))
plt.plot(curve_df["threshold"], curve_df["net_profit"], color="#10B981", lw=3, label="Campaign Net Profit ($)")
plt.axvline(x=opt_results["optimal_threshold"], color="#0D9488", linestyle="--", label=f"Optimal Threshold ({opt_results['optimal_threshold']:.3f})")
plt.axvline(x=0.50, color="#EF4444", linestyle=":", label="Default Cutoff (0.50)")
plt.title("Cost-Benefit Matrix: Campaign Net Profit as a Function of Classification Cutoff")
plt.xlabel("Decision Threshold (tau)")
plt.ylabel("Net Campaign Profit ($)")
plt.legend()
plt.show()

## 3. Marketing Budget Allocation (Knapsack LP Formulation)
Solving the bounded Knapsack optimization problem with `scipy.optimize.linprog`.

In [ ]:
budget_engine = CampaignBudgetOptimizer(cost_per_call=15.0, conversion_revenue=300.0)
all_probs = model.predict_proba(pipeline.transform(X))[:, 1]

sim_budget = 20000.0
lp_res = budget_engine.solve_budget_allocation_lp(df_raw, all_probs, budget_limit=sim_budget)

print(f"Budget: ${sim_budget:,.2f}")
print(f"Contacts Selected: {lp_res['total_contacts_selected']:,}")
print(f"Expected Conversions: {lp_res['expected_conversions']:.0f}")
print(f"Expected Net Profit: ${lp_res['expected_net_profit']:,.2f}")
print(f"Campaign ROI: {lp_res['expected_roi_percent']:.1f}%")